# Credit Default Classification

## Background and Problem

## Data Description

## Preparation and Feature Generation

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression

# Load data
default = pd.read_csv('default.csv')
default.columns = default.iloc[0]
default = default[1:]
default.reset_index(drop = True, inplace = True)
default = default.apply(pd.to_numeric, errors = 'coerce')

# Feature engineering
default['LATE_MONTHS'] = (default[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']] > 0).sum(axis = 1)

default['AVG_BILL'] = (default['BILL_AMT1'] + default['BILL_AMT2'] + default['BILL_AMT3'] + default['BILL_AMT4'] + default['BILL_AMT5'] + default['BILL_AMT6']) / 6

default['AVG_UTILIZATION'] = default['AVG_BILL'] / default['LIMIT_BAL']

default['AVG_PAYMENT'] = (default['PAY_AMT1'] + default['PAY_AMT2'] + default['PAY_AMT3'] + default['PAY_AMT4'] + default['PAY_AMT5'] + default['PAY_AMT6']) / 6

default['MAX_DELAY'] = default[['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']].max(axis = 1)

default['RECENT_LATE'] = (default[['PAY_0', 'PAY_2', 'PAY_3']] > 0).sum(axis = 1)

default['PAYMENT_RATIO'] = default['AVG_PAYMENT'] / default['AVG_BILL'].clip(lower = 0).replace(0, np.nan)

default['MAX_UTILIZATION'] = default[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].max(axis = 1) / default['LIMIT_BAL'].replace(0, np.nan)

# Clean categorical codes
default['EDUCATION'] = default['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
default['MARRIAGE'] = default['MARRIAGE'].replace({0: 3})

# Remove ID
default = default.drop(columns = ['ID'])

# Separate predictors and response
X = default.drop(columns = ['default payment next month'])
y = default['default payment next month']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

# One-hot encode categorical variables
X_train = pd.get_dummies(X_train, columns = ['SEX', 'EDUCATION', 'MARRIAGE'], dtype = int)
X_test = pd.get_dummies(X_test, columns = ['SEX', 'EDUCATION', 'MARRIAGE'], dtype = int)
X_test = X_test.reindex(columns = X_train.columns, fill_value = 0)

# Handle missing and infinite values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
imputer = SimpleImputer(strategy = 'median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(imputer.transform(X_test), columns = X_test.columns, index = X_test.index)

# Standardize predictors
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns = X_train.columns, index = X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns = X_test.columns, index = X_test.index)

## Model 1: Logistic Regression

## Model 2: Support Vector Machine

## Model 3: Perceptron Classifier

## Comparison and Evaluation

## Conclusion

## References